# A two-node mixed feedback loop, end to end

This notebook takes the smallest interesting regulatory network, walks through what
DSGRN and `DSGRN_utils` produce for it, and writes a JSON database covering **every**
parameter node in the parameter graph. That database is what the DSGRN visualization
tool at <https://chomp.rutgers.edu/projects/dsgrn_viz/index.html> consumes
(Appendix, *DSGRN Visualization Tool*).

The network is the **mixed feedback loop**: one activation and one repression around a
two-cycle. It has nine parameter nodes, so the whole database fits in a few kilobytes
and every node can be inspected by hand.

> **Which algorithm is this?** `DSGRN_utils` here implements the definitions of
> `Rook_Field_Paper_v2`. Passing `legacy=True` to `ConleyMorseGraph` or
> `save_morse_graph_database_json` restores the earlier behaviour, which is what
> reproduces figures made before the definitions were aligned. See
> `rookfields/reports/FINDINGS.md`.

## 0. Setup

On Colab, install the two compiled dependencies and this repository. Locally, this
cell does nothing if the packages already import.

In [ ]:
import importlib.util, subprocess, sys

try:                      # find_spec raises when the parent package is absent,
    import google.colab   # so detect Colab by import rather than by spec
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# The fork of DSGRN_utils carrying the manuscript's definitions.
# Point this at wherever the patched package is hosted.
DSGRN_UTILS_REPO = 'https://github.com/bernardorivas/RookFields'

def ensure(mod, *pip_args):
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pip_args])

if IN_COLAB:
    ensure('DSGRN', 'DSGRN')
    ensure('pychomp', 'pyCHomP2')
    ensure('DSGRN_utils', f'git+{DSGRN_UTILS_REPO}#subdirectory=DSGRN_utils')

import DSGRN
import DSGRN_utils
print('DSGRN and DSGRN_utils ready')

## 1. The network

A DSGRN network is given by one line per node, `name : algebraic_expression`, where
`m` is activation by node `m` and `~m` is repression. Here `y` activates nothing and
represses `x`, while `x` activates `y`: a single loop of net negative sign, with one
edge of each type.

In [ ]:
net_spec = """
x : (~y)
y : (x)
"""

network = DSGRN.Network(net_spec)
print(network.specification())

for n in range(network.size()):
    print(f'node {n} = {network.name(n)!r}: '
          f'inputs {[network.name(m) for m in network.inputs(n)]}, '
          f'outputs {[network.name(m) for m in network.outputs(n)]}')

DSGRN.DrawGraph(network)

## 2. The parameter graph

DSGRN partitions the (continuous) parameter space into finitely many semi-algebraic
regions; the *parameter graph* has one node per region and an edge for each
codimension-one adjacency. Each node fixes the ordering of production values against
decay thresholds, which is all the combinatorial machinery needs.

In [ ]:
parameter_graph = DSGRN.ParameterGraph(network)
print('parameter nodes:', parameter_graph.size())

par_index = 4
parameter = parameter_graph.parameter(par_index)
print()
print(f'node {par_index} inequalities:')
print(parameter.partialorders())
print('adjacent nodes:', parameter_graph.adjacencies(par_index))

## 3. One parameter: Morse graph and Morse sets

`ConleyMorseGraph` runs the whole pipeline: wall labelling to rook field to the
multivalued map $\mathcal{F}_i$, then strongly connected components, the Conley
complex, and the Morse graph.

`level` selects the map: `1` uses only the local wall conditions, `2` adds the
decision walls, `3` adds the regulation-cycle rules, `4` uses the relaxed decision
wall. In two dimensions `level=3` already resolves every double edge.

In [ ]:
morse_graph, stg, graded_complex = DSGRN_utils.ConleyMorseGraph(parameter, level=3)

for v in morse_graph.vertices():
    index, cells, conley = morse_graph.vertex_label(v)
    print(f'Morse node {index}: Conley index {tuple(conley)} over Z_2, {cells} cell(s)')

# 'c' shows the Conley index, 'a' the FP / PO / T / M classification
DSGRN_utils.PlotMorseGraph(morse_graph, label='c:a')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
DSGRN_utils.PlotMorseSets(morse_graph, stg, graded_complex, ax=ax)
ax.set_title(f'Morse sets of $\\mathcal{{F}}_3$ at parameter node {par_index}')
plt.show()

Blue arrows are single edges of the state transition graph, red arrows are double
edges (the map could not orient that wall), and a red disc marks a self-edge at an
equilibrium cell. A component with no double edges and no self-edge is one the
combinatorics has fully resolved.

## 4. Every parameter node at once

Nine nodes is small enough to tabulate the whole parameter graph.

In [ ]:
import collections

def classify(conley):
    """The FP / PO / T / M reading of a Conley index, as PlotMorseGraph uses."""
    nz = [k for k, b in enumerate(conley) if b]
    if not nz:
        return 'T'
    if len(nz) == 1 and conley[nz[0]] == 1:
        return f'FP({nz[0]})'
    if len(nz) == 2 and nz[1] == nz[0] + 1 and conley[nz[0]] == conley[nz[1]] == 1:
        return 'PO'
    return 'M'

rows = []
for i in range(parameter_graph.size()):
    mg, _stg, _gc = DSGRN_utils.ConleyMorseGraph(parameter_graph.parameter(i), level=3)
    labels = [mg.vertex_label(v) for v in mg.vertices()]
    rows.append((i, len(labels),
                 ', '.join(sorted(classify(tuple(l[2])) for l in labels))))

print(f'{"node":>5}  {"Morse nodes":>11}  classification')
for i, n, kinds in rows:
    print(f'{i:>5}  {n:>11}  {kinds}')
print()
print('distinct Morse graphs by classification:',
      len(collections.Counter(r[2] for r in rows)))

## 5. Save the JSON database for all parameters

`save_morse_graph_database_json` writes one file holding the network, the geometry of
the blowup cell complex, the parameter graph, and the Morse graph plus Morse sets for
every parameter node requested. Omitting `param_indices` covers the entire parameter
graph.

In [ ]:
import json, os

database = 'mixed_feedback_2d.json'

DSGRN_utils.save_morse_graph_database_json(
    network,
    database,
    param_indices=None,   # None means every node of the parameter graph
    level=3,
)

print(f'wrote {database}  ({os.path.getsize(database)} bytes)')

### What is in the file

In [ ]:
with open(database) as fh:
    data = json.load(fh)

for key, value in data.items():
    print(f'{key:20s} {type(value).__name__:5s} {len(value)} entries')

print()
print('network:          ', data['network'].keys())
print('complex:          ', data['complex'].keys())
print('parameter_graph:  ', data['parameter_graph'].keys())
print()
entry = data['dynamics_database'][par_index]
print(f'dynamics_database[{par_index}] keys:', list(entry))
print(json.dumps(entry['morse_graph'], indent=2)[:400], '...')

## 6. Viewing it

Open <https://chomp.rutgers.edu/projects/dsgrn_viz/index.html> and upload the file.
The top panel shows the network, the Morse graph, and the cell complex with the Morse
sets highlighted; the bottom panel shows the parameter graph, and clicking a node
switches the view to that parameter.

In Colab, download it with:

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(database)

## 7. Scaling up

The same three lines work for any network; only the size of the parameter graph
changes. Adding a self-activation to `x` takes the mixed feedback loop from 9 nodes to
120, which is still quick to export in full.

Beyond a few thousand nodes, pass an explicit `param_indices` list rather than `None` —
the database holds a Morse graph per node and grows linearly.

In [ ]:
bigger_spec = """
x : (x)(~y)
y : (x)
"""

bigger = DSGRN.Network(bigger_spec)
bigger_pg = DSGRN.ParameterGraph(bigger)
print('parameter nodes:', bigger_pg.size())

DSGRN_utils.save_morse_graph_database_json(bigger, 'mixed_feedback_selfact_2d.json', level=3)
print('wrote mixed_feedback_selfact_2d.json',
      f'({os.path.getsize("mixed_feedback_selfact_2d.json")} bytes)')